**🪖  Unified Military Analytics**

# Milestone 1: Data Collection and Preparation

This notebook implements Milestone 1 of the project.

The objective is to collect military capability data for multiple countries, structure it into a dataset, and prepare it for further analysis.

## Module 1: Scraping Setup and Execution

In this module, we build a web scraping pipeline to collect military capability metrics.

### Step 1: Create Data Folder

This step creates a local folder named data to store all scraped country CSV files.

In [1]:
import os
os.makedirs("data", exist_ok=True)

### Step 2: Mount Google Drive

This step mounts Google Drive to enable saving files for persistent storage when using Google Colab.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Import Required Libraries

We import the necessary libraries required for web scraping and data processing.

In [3]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

### Step 4: Load Country URL

The country URL is defined. Each URL corresponds to a page containing military capability metrics.

In [4]:
url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=india"

### Step 5: Extract Military Data

This step sends a request to the Global Firepower website and extracts India's military data, including power index and detailed metrics.

In [5]:
response = requests.get(url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

data = {}

title = soup.find("h1")
data["country"] = title.get_text(strip=True)

overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)
    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None

metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

### Step 6: Convert Data into DataFrame

The extracted data is converted into a structured DataFrame and saved as a CSV file.

In [6]:
print("India Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")

df = pd.DataFrame([data])
df.to_csv("data/india_test.csv", index=False)

print("Saved to data/india_test.csv")

India Data Extracted:

country: 2026India Military Strength
power_index_rank: 4
power_index_score: 0.1346
purchasing_power_parity: 3
foreign_exchange/gold: 5
defense_budget: 5
external_debt: 110
square_land_area: 7
coastline_coverage: 91
shared_borders: 129
waterways_(usable): 10
total_population: 2
available_manpower: 2
fit-for-service: 2
reaching_mil_age_annually: 1
tot_mil._personnel_(est.): 4,958,000
active_personnel: 2
reserve_personnel: 8
paramilitary: 2
air_force_personnel*: 3
army_personnel*: 3
navy_personnel*: 4
yearly_mobilization_potential: 23,652,761
aircraft_total: 4
fighters: 4
attack_types: 4
transports_(fixed-wing): 4
trainers: 8
special-mission: 5
tanker_fleet: 11
helicopters: 4
attack_helicopters: 9
tanks: 5
vehicles: 2
self-propelled_artillery: 35
towed_artillery: 3
mlrs_(rocket_artillery): 16
total_assets: 4
total_tonnage: 5
aircraft_carriers: 3
helicopter_carriers: 145
destroyers: 5
frigates: 3
corvettes: 5
submarines: 7
patrol_vessels: 7
mine_warfare: 145
oil_prod

### Step 7: Automate Scraping for Multiple Countries

This step reads multiple URLs from a file and applies the same scraping logic to collect data for all countries.

In [34]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

base_url = "https://www.globalfirepower.com/"

# Step 1: Get all country links automatically
main_page = requests.get(base_url, timeout=15)
soup = BeautifulSoup(main_page.text, "lxml")

country_links = []

for link in soup.find_all("a", href=True):
    href = link["href"]
    if "country-military-strength-detail.php?country_id=" in href:
        full_link = base_url + href
        if full_link not in country_links:
            country_links.append(full_link)

print("Total countries found:", len(country_links))


# Step 2: Scrape each country
for i, url in enumerate(country_links):
    try:
        print(f"Processing {i+1}/{len(country_links)}")

        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")
        data = {}

        # Country Name
        title = soup.find("h1")
        if title is None:
            continue

        data["country"] = title.get_text(strip=True)

        # Power Index
        overview_text = soup.find("span", class_="textNormal")

        if overview_text:
            text = overview_text.get_text(" ", strip=True)
            rank_match = re.search(r"ranked\s+(\d+)", text)
            score_match = re.search(r"score of\s+([\d.]+)", text)

            data["power_index_rank"] = rank_match.group(1) if rank_match else None
            data["power_index_score"] = score_match.group(1) if score_match else None

        # Metrics
        metric_blocks = soup.find_all("div", class_="specsGenContainers")

        for block in metric_blocks:
            label = block.find("span", class_="textLarge")
            value = block.find("span", class_="textWhite")

            if label and value:
                key = (
                    label.get_text(strip=True)
                    .replace(":", "")
                    .lower()
                    .replace(" ", "_")
                )
                data[key] = value.get_text(strip=True)

        # Save CSV
        df = pd.DataFrame([data])
        df.to_csv(f"data/country_{i}.csv", index=False)

        time.sleep(2)  # Prevent blocking

    except Exception as e:
        print(f"Error: {url}")

print("✅ All countries scraped successfully!")

Total countries found: 93
Processing 1/93
Processing 2/93
Processing 3/93
Processing 4/93
Processing 5/93
Processing 6/93
Processing 7/93
Processing 8/93
Processing 9/93
Processing 10/93
Processing 11/93
Processing 12/93
Processing 13/93
Processing 14/93
Processing 15/93
Processing 16/93
Processing 17/93
Processing 18/93
Processing 19/93
Processing 20/93
Processing 21/93
Processing 22/93
Processing 23/93
Processing 24/93
Processing 25/93
Processing 26/93
Processing 27/93
Processing 28/93
Processing 29/93
Processing 30/93
Processing 31/93
Processing 32/93
Processing 33/93
Processing 34/93
Processing 35/93
Processing 36/93
Processing 37/93
Processing 38/93
Processing 39/93
Processing 40/93
Processing 41/93
Processing 42/93
Processing 43/93
Processing 44/93
Processing 45/93
Processing 46/93
Processing 47/93
Processing 48/93
Processing 49/93
Processing 50/93
Processing 51/93
Processing 52/93
Processing 53/93
Processing 54/93
Processing 55/93
Processing 56/93
Processing 57/93
Processing 58/

### Step 8: Merge All CSV Files

This step combines all individual country datasets into a single dataset.

In [37]:
import pandas as pd
import os

folder_path = "data"

all_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

df_list = []

for file in all_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)

combined_df.head()

,country,power_index_rank,power_index_score,purchasing_power_parity,foreign_exchange/gold,defense_budget,external_debt,square_land_area,coastline_coverage,shared_borders,...,coal_deficit,coal_proven_reserves,internet_coverage,labor_force,merchant_marine_fleet,ports_/_trade_terminals,airports,roadway_coverage,railway_coverage,total_tonnage
0,2026Estonia Military Strength,106,2.3201,115,120,75,72,123,82,13,...,-800 mt,145,8,138,79,28,83,58,84,NaN
1,2026Tunisia Military Strength,79,1.7823,81,82,76,64,88,45,30,...,"-2,000 mt",145,26,92,80,31,97,97,71,NaN
2,2026Germany Military Strength,12,0.2463,6,12,4,142,61,64,79,...,"-31,253,000 mt",6,7,15,31,21,9,10,6,15.0
3,2026Ecuador Military Strength,72,1.4479,67,89,65,80,72,63,54,...,"-14,000 mt",72,23,61,56,41,21,89,90,NaN
4,2026Colombia Military Strength,43,0.7551,32,41,28,100,25,77,114,...,"+42,656,000 mt",19,23,26,57,33,11,25,72,46.0


### Step 9: Export Final Dataset

This step exports the combined dataset for further processing.

In [38]:
combined_df.to_csv("Military_raw_data.csv", index=False)
print("Combined raw dataset saved as Military_raw_data.csv")

Combined raw dataset saved as Military_raw_data.csv
